In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix

print("--- FASE 1: CARREGAMENTO ---")

# ATENÇÃO: Se usar o Colab, este caminho NÃO FUNCIONARÁ.
# Este caminho é para o Jupyter Notebook/ambiente local no seu PC.
BASE_PATH = "C:\\Users\\User\\Desktop\\escola\\Projeto\\PlaySafe4All\\PlaySafe4All\\Data"

# Construindo os caminhos completos
FILE_PATH_1 = f"{BASE_PATH}\\Base Artigo 1.csv"
FILE_PATH_2 = f"{BASE_PATH}\\Base Artigo 2.csv"
FILE_PATH_3 = f"{BASE_PATH}\\Base Artigo 3.csv"

# Inicializar os dataframes
df1, df2, df3 = None, None, None
load_successful = False

# Carregar os três arquivos CSV
try:
    # O r antes das aspas é para tratar as barras invertidas do Windows
    df1 = pd.read_csv(FILE_PATH_1, sep=',')
    df2 = pd.read_csv(FILE_PATH_2, sep=',')
    df3 = pd.read_csv(FILE_PATH_3, sep=',')
    print("Arquivos carregados com sucesso.")
    load_successful = True
except FileNotFoundError:
    print("ERRO CRÍTICO: Não foi possível encontrar os ficheiros CSV. Verifique se o caminho completo no código está correto.")

# Mostrar informações básicas (apenas se o carregamento foi bem sucedido)
if load_successful:
    print(f"Total de amostras brutas: {len(df1) + len(df2) + len(df3)}")
    print(f"Artigo 1 shape: {df1.shape}")
    print(f"Artigo 2 shape: {df2.shape}")
    print(f"Artigo 3 shape: {df3.shape}")

--- FASE 1: CARREGAMENTO ---
Arquivos carregados com sucesso.
Total de amostras brutas: 181
Artigo 1 shape: (60, 233)
Artigo 2 shape: (61, 173)
Artigo 3 shape: (60, 341)


In [10]:
# ==============================================================================
# CÉLULA 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES (INCLUI SPLIT E SCALE)
# ==============================================================================

print("\n--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES ---")

# 1. UNIÃO E FILTRAGEM DE FEATURES
df_combined = pd.concat([df1, df2, df3], ignore_index=True)

if 'Numero' in df_combined.columns:
    df_combined.drop(columns=['Numero'], inplace=True)

TARGET_COL = 'Entorse'

# LISTA FINAL E COMPLETA DE FEATURES SELECIONADAS E JUSTIFICADAS (Versão da imagem)
COLUNAS_SELECIONADAS = [
    TARGET_COL,
    #'Posição',
    #'M.I.Dominante',
    #'Lado_entorse',
    #'T0IMC',
    #'T0MassaGordaPerc',
    #dc'T0MassaMagra',
    #Fatores de Exposição/Carga (Carga Aguda)
    'T0_T1_Match_Time_exposure',
    'T0_T1_Training_Time_exposure',
    # Fatores de Risco Biomecânico (Performance)
    'T0SRTMax',           # Flexibilidade (Senta e Alcança)
    'T0TTestMin',         # Agilidade (T-Test)
    'T0SJmMax',           # Força Explosiva (Squat Jump)
    'T0Veli',             # Velocidade (30m)
    'T0RazaoAADto',       # Rácio de Força Isocinética Direito
    'T0RazaoAAEsq',       # Rácio de Força Isocinética Esquerdo
]

# Filtra o DataFrame combinado para incluir apenas as colunas que existem nas bases
cols_to_keep = list(set(COLUNAS_SELECIONADAS) & set(df_combined.columns))
df_filtered = df_combined[cols_to_keep].copy()

# 2. TRATAMENTO DE VALORES OMISSOS (Imputação)
df_filtered[TARGET_COL].fillna(0, inplace=True) 
df_filtered[TARGET_COL] = df_filtered[TARGET_COL].astype(int)

# Identifica colunas
numerical_cols = df_filtered.select_dtypes(include=np.number).columns.tolist()
if TARGET_COL in numerical_cols:
    numerical_cols.remove(TARGET_COL)
categorical_cols = df_filtered.select_dtypes(include=['object']).columns.tolist()

# Imputação de NaN (Median para numéricas, Mode para categóricas)
for col in numerical_cols:
    df_filtered[col] = df_filtered[col].fillna(df_filtered[col].median())

for col in categorical_cols:
    if not df_filtered[col].mode().empty:
        df_filtered[col] = df_filtered[col].fillna(df_filtered[col].mode()[0])

# 3. DEFINIÇÃO DE X/Y E CODIFICAÇÃO CATEGÓRICA
Y = df_filtered[TARGET_COL] 
X = df_filtered.drop(columns=[TARGET_COL]) 
X = pd.get_dummies(X, drop_first=True) 

print(f"Dimensão de X após codificação: {X.shape}")

# 4. DIVISÃO TREINO/TESTE E NORMALIZAÇÃO
# Divide 25% para treino, 75% para teste, estratificado
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.75, random_state=42, stratify=Y)

# Normalização (StandardScaler)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("--- ✅ FASE 2 CONCLUÍDA! DADOS PRONTOS. ---")


--- FASE 2: PRÉ-PROCESSAMENTO E SELEÇÃO DE FEATURES ---
Dimensão de X após codificação: (181, 56)
--- ✅ FASE 2 CONCLUÍDA! DADOS PRONTOS. ---


C:\Users\User\AppData\Local\Temp\ipykernel_2564\2949368200.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filtered[TARGET_COL].fillna(0, inplace=True)


In [11]:
# ==============================================================================
# CÉLULA 3: TREINO DO MODELO COM RANDOM FOREST
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier

print("\n--- FASE 3: CONSTRUÇÃO E TREINO DO MODELO RANDOM FOREST ---")

# 1. CONSTRUÇÃO DO MODELO RANDOM FOREST
model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    max_depth=5
)

# 2. TREINO DO MODELO
model_rf.fit(X_train_scaled.values, y_train)

print("\n--- ✅ FASE 3 CONCLUÍDA! MODELO TREINADO. ---")


--- FASE 3: CONSTRUÇÃO E TREINO DO MODELO RANDOM FOREST ---

--- ✅ FASE 3 CONCLUÍDA! MODELO TREINADO. ---


In [12]:
# ==============================================================================
# CÉLULA 4: AVALIAÇÃO FINAL DO DESEMPENHO (RANDOM FOREST)
# ==============================================================================
from sklearn.metrics import recall_score, f1_score, accuracy_score, confusion_matrix
import time

print("\n--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---")

# ⏱️ Início da medição do tempo
start_time = time.perf_counter()

# 1. PREVISÃO NOS DADOS DE TESTE (SÓ FEATURES!)
y_pred = model_rf.predict(X_test_scaled.values)

# 2. CÁLCULO DAS MÉTRICAS
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f_score = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

# ⏱️ Fim da medição do tempo
end_time = time.perf_counter()
execution_time_ms = (end_time - start_time) * 1000

print("\n=== RESULTADOS FINAIS DE DESEMPENHO ===")
print(f"Precisão (Accuracy): {accuracy:.4f}")
print(f"Recall (Sensibilidade): {recall:.4f}")
print(f"F-Score: {f_score:.4f}")
print(f"Tempo de Execução: {execution_time_ms:.2f} ms")

print("\nMatriz de Confusão:")
print(conf_matrix)

# 3. FALSOS NEGATIVOS (ERRO CRÍTICO)
if conf_matrix.shape == (2, 2):
    fn = conf_matrix[1, 0]
    # Este FN é o número que o professor quer ver a ZERO
    print(f"\nFalsos Negativos (FN - Entorse real não prevista): {fn}")

print("\n--- ✅ PROJETO CONCLUÍDO E VALIDADO (RANDOM FOREST) ---")



--- FASE 4: AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (RANDOM FOREST) ---

=== RESULTADOS FINAIS DE DESEMPENHO ===
Precisão (Accuracy): 0.5882
Recall (Sensibilidade): 0.5902
F-Score: 0.5625
Tempo de Execução: 18.64 ms

Matriz de Confusão:
[[36 25]
 [31 44]]

Falsos Negativos (FN - Entorse real não prevista): 31

--- ✅ PROJETO CONCLUÍDO E VALIDADO (RANDOM FOREST) ---
